# EDA Vision Deepfake

Exploratory analysis for vision/deepfake datasets.

Steps:
- Inventory train_real/train_fake and face emotion folders.
- Summarize image counts by class and extension.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from collections import Counter
from pathlib import Path

vision_root = REPO_ROOT / 'data' / 'raw' / 'vision'
summary = {
    'vision_root': str(vision_root),
    'deepfake_counts': {},
    'face_emotion_counts': {},
    'extensions': {},
}

image_exts = {'.jpg', '.jpeg', '.png', '.bmp'}

print('Vision root:', vision_root)
if not vision_root.exists():
    print('Missing:', vision_root)
else:
    for child in sorted(vision_root.iterdir()):
        if child.is_dir():
            print(' -', child.name)

    for name in ['train_real', 'train_fake']:
        path = vision_root / name
        if not path.exists():
            print('Missing:', path)
            continue
        files = [p for p in path.rglob('*') if p.is_file() and p.suffix.lower() in image_exts]
        summary['deepfake_counts'][name] = len(files)
        ext_counts = Counter(p.suffix.lower() for p in files)
        summary['extensions'][name] = dict(ext_counts)
        print(f'{name}: {len(files)} images')

    face_emotion_dir = vision_root / 'face_emotion'
    if face_emotion_dir.exists():
        for class_dir in sorted(face_emotion_dir.iterdir()):
            if not class_dir.is_dir():
                continue
            files = [p for p in class_dir.rglob('*') if p.is_file() and p.suffix.lower() in image_exts]
            summary['face_emotion_counts'][class_dir.name] = len(files)
            print(f'face_emotion/{class_dir.name}: {len(files)} images')


In [ ]:
# Sample a few image files for sanity check.
if vision_root.exists():
    samples = []
    for folder in ['train_real', 'train_fake']:
        path = vision_root / folder
        if not path.exists():
            continue
        for img in path.rglob('*'):
            if img.is_file() and img.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                samples.append(str(img.relative_to(REPO_ROOT)))
            if len(samples) >= 10:
                break
        if len(samples) >= 10:
            break
    print('Sample images:')
    for item in samples:
        print(' -', item)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_vision_deepfake_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize vision-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'vision' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No vision entries found in TRAINING_DATA.json')
    else:
        print('vision datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
